# Lab 3 — Confusion Matrix & Classification Metrics

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Build and read a **confusion matrix** (rows = true, cols = predicted).
2. Label **TP, FP, TN, FN** for loan default prediction.
3. Compute **accuracy, precision, recall, F1**.
4. Explore how changing the **decision threshold** shifts precision vs recall.

> **Checkpoints:** CM `[[63,40],[42,55]]` · accuracy ≈ **0.59** · F1 ≈ **0.57**



## Confusion matrix layout (binary classification)

**Positive class = default (1)**

```
                 Predicted
                 0      1
Actual  0       TN     FP
        1       FN     TP
```

| Cell | Meaning | Business impact |
|------|---------|-----------------|
| **TP** | Correctly flagged default | Good — catch risky loan |
| **TN** | Correctly cleared safe loan | Good — approve good borrower |
| **FP** | False alarm (predicted default, was fine) | Lost revenue / unnecessary review |
| **FN** | Missed default | **Costly** — loan goes bad |


---

## 1. Train model (same setup as Lab 2)


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
X, y = df[NUMERIC_FEATURES], df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"test size: {len(y_test)}")


---

## 2. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Lab 3 — Confusion matrix")
print("rows=true, cols=pred:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion matrix — loan default")
plt.tight_layout()
plt.show()


---

## 3. Classification metrics

| Metric | Formula (default=positive) | Question answered |
|--------|---------------------------|-------------------|
| **Accuracy** | (TP+TN) / total | Overall correct? |
| **Precision** | TP / (TP+FP) | Of predicted defaults, how many real? |
| **Recall** | TP / (TP+FN) | Of actual defaults, how many caught? |
| **F1** | Harmonic mean of P & R | Balance precision and recall |


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

metrics_df = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "F1"],
    "value": [accuracy, precision, recall, f1],
})
display(metrics_df.round(4))

# Manual precision check
manual_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
print(f"manual precision: {manual_precision:.4f}")


---

## 4. `classification_report`


In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))


**Support** = number of true instances per class in the test set (103 non-default, 97 default).


---

## 5. Threshold experiment — precision vs recall trade-off

Default threshold = **0.5**. Lower threshold → more predicted defaults → **higher recall**, often **lower precision**.


In [ ]:
def metrics_at_threshold(threshold):
    pred_t = (y_proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
    }

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
thresh_df = pd.DataFrame([metrics_at_threshold(t) for t in thresholds])
display(thresh_df.round(4))


In [ ]:
# Highlight threshold 0.4 (lab exercise)
pred_04 = (y_proba >= 0.4).astype(int)
print("At threshold 0.4:")
print(f"  precision: {precision_score(y_test, pred_04, zero_division=0):.4f}")
print(f"  recall:    {recall_score(y_test, pred_04, zero_division=0):.4f}")
print(f"  F1:        {f1_score(y_test, pred_04, zero_division=0):.4f}")


**Credit risk framing:** Lower threshold catches more defaults (recall↑) but flags more good loans (precision↓). The "right" threshold depends on business cost of FN vs FP.


---

## 6. Checkpoint summary


In [ ]:
assert cm.shape == (2, 2)
assert tn > 0 and fp > 0 and fn > 0 and tp > 0
assert abs(accuracy - 0.59) < 0.02
assert abs(f1 - 0.5729) < 0.02

print("✓ All checkpoint assertions passed")
print(f"accuracy: {accuracy:.4f}, F1: {f1:.4f}")


---

## Reflection questions

1. Which error is worse for a lender — FP or FN? Why?
2. Why is accuracy alone insufficient on Day 6 fraud data (99:1 imbalance)?
3. What metric does Lab 4 (ROC/AUC) add that threshold-based metrics miss?

**Previous:** [Lab 2 — Logistic regression](lab02_logistic_regression.ipynb)  
**Next:** [Lab 4 — ROC and AUC](lab04_roc_auc.ipynb)
